# Deterministic draft simulation

- **Question:** If the user selects a candidate now, how does that choice affect a legal final roster across plausible rest-of-draft paths?
- **Data:** A small, fully mapped **synthetic fixture** created in this notebook, exact league rules, versioned engine assumptions, and an event-sourced draft state. Live player rows are not used in the simulation.
- **Unit of observation:** One candidate and one simulated remainder of a snake draft.
- **Target:** Final legal starter value plus the configured bench-depth credit; not wins, playoffs, or championships.
- **Cutoff:** A session freezes its rules, projection run, ADP build, canonical player pool, engine configuration, seed, and simulation count before picks are evaluated.
- **Validation:** Deterministic event replay, stable-seed repetition, input-order invariance, legal FLEX/SUPERFLEX assignment, and recomputable score components.
- **Interpretation caveat:** Opponent demand, positional runs, and ADP availability are transparent uncalibrated assumptions. The output is a baseline draft recommendation score, never a championship probability.

## Keep the live identity gate separate

The validated live checkpoint has 246 FFC market rows and 0 reviewed canonical mappings. The next cell reads that status only. It never joins a live player to a projection and never uses display name as an identity key. The executable simulation later in the notebook uses a separate, clearly labeled synthetic pool whose canonical mappings are complete.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from datetime import UTC, datetime

import pandas as pd

from fantasy_draft_ai.config import find_project_root, load_config
from fantasy_draft_ai.services.adp_market import load_adp_market_board

PROJECT_ROOT = find_project_root()
app_config = load_config()
live_market = load_adp_market_board(app_config)
live_rows = len(live_market.rows)
live_mapped = sum(row.identity.player_id is not None for row in live_market.rows)
live_gate = pd.DataFrame(
    [
        ("validated live market rows", live_rows),
        ("reviewed canonical mappings", live_mapped),
        ("full market simulation ready", bool(live_rows and live_mapped == live_rows)),
        ("identity rule", "canonical player_id only; never display name"),
    ],
    columns=["gate", "value"],
)
live_gate

## Exact FLEX and SUPERFLEX assignment

Flexible slots are a matching problem, not independent position counts. The assignment below has direct QB/RB/WR/TE slots, one FLEX, and one SUPERFLEX. The rules engine finds the maximum-value legal set and reports every concrete slot.

In [ ]:
from fantasy_draft_ai.draft.roster import RosterPlayer, assign_roster
from fantasy_draft_ai.rules.models import DraftSettings, FlexSlot, LeagueRules
from fantasy_draft_ai.scoring.engine import ScoringRules

assignment_rules = LeagueRules(
    season=2026,
    teams=4,
    draft=DraftSettings(rounds=6),
    starters={"QB": 1, "RB": 1, "WR": 1, "TE": 1},
    flex_slots=(
        FlexSlot(name="FLEX", count=1, eligible=("RB", "WR", "TE")),
        FlexSlot(
            name="SUPERFLEX", count=1, eligible=("QB", "RB", "WR", "TE")
        ),
    ),
    bench=0,
    scoring=ScoringRules(reception=1),
)
assignment = assign_roster(
    [
        RosterPlayer("QB-A", "QB", 310),
        RosterPlayer("QB-B", "QB", 290),
        RosterPlayer("RB-A", "RB", 280),
        RosterPlayer("RB-B", "RB", 250),
        RosterPlayer("WR-A", "WR", 270),
        RosterPlayer("TE-A", "TE", 260),
    ],
    assignment_rules,
)
assert assignment.legal and assignment.starter_coverage == 1.0
pd.DataFrame(
    [
        {
            "slot": item.slot,
            "player_id": item.player.player_id,
            "position": item.player.position,
            "projected_points": item.player.projected_points,
        }
        for item in assignment.starters
    ]
).sort_values("slot", ignore_index=True)

## Build a fully mapped synthetic fixture

Every row below has a made-up canonical ID, projection interval, ADP location/scale, reviewed mapping label, and timestamp. These values are teaching data only. No synthetic row is written to DuckDB or used for production training. The small engine configuration uses 16 paths so the notebook executes quickly.

In [ ]:
from fantasy_draft_ai.draft.pool import FrozenDraftPlayer, player_pool_fingerprint
from fantasy_draft_ai.recommendations.config import load_draft_engine_config

engine_config = load_draft_engine_config().model_copy(
    update={
        "default_simulations": 16,
        "maximum_simulations": 16,
        "candidate_count": 4,
        "opponent_candidate_window": 24,
        "work_budget": 10_000,
    }
)
fixture_rules = LeagueRules(
    season=2026,
    teams=4,
    draft=DraftSettings(rounds=3),
    starters={"WR": 2},
    flex_slots=(FlexSlot(name="FLEX", count=1, eligible=("RB", "WR")),),
    bench=0,
    scoring=ScoringRules(reception=1),
)


def fixture_player(
    player_id: str, position: str, p50: float, adp: float, downside: float, upside: float
) -> FrozenDraftPlayer:
    return FrozenDraftPlayer(
        player_id=player_id,
        display_name=f"Synthetic {player_id}",
        position=position,
        p10=p50 - downside,
        p50=p50,
        p90=p50 + upside,
        prediction_status="synthetic_interval_fixture",
        projection_source="synthetic_fixture",
        projection_method="notebook_only",
        market_source="synthetic_market",
        market_snapshot_id="synthetic-2026-08-01",
        market_captured_at=datetime(2026, 8, 1, tzinfo=UTC),
        average_pick=adp,
        availability_scale=4.0,
        availability_evidence="synthetic_notebook_standard_deviation",
        mapping_confidence="reviewed",
    )


fixture_players = tuple(
    [
        *(
            fixture_player(
                f"WR-{index + 1:02d}",
                "WR",
                310.0 - 6.0 * index,
                float(index + 1),
                8.0 + 3.0 * (index % 4),
                12.0 + 4.0 * ((index + 1) % 5),
            )
            for index in range(14)
        ),
        *(
            fixture_player(
                f"RB-{index + 1:02d}",
                "RB",
                225.0 - 5.0 * index,
                float(index + 15),
                10.0 + 2.0 * (index % 3),
                14.0 + 3.0 * ((index + 1) % 4),
            )
            for index in range(10)
        ),
    ]
)
assert all(player.has_market_evidence for player in fixture_players)
print({"synthetic_rows": len(fixture_players), "all_mapped": True})

## Replay the event stream

The start event freezes every input fingerprint. Two opponent picks are appended with prior/result fingerprint links. Replaying those immutable events must return the same state with the user on the clock at overall pick 3.

In [ ]:
from fantasy_draft_ai.draft.state import DraftEvent, apply_event, replay_events

EVENT_TIME = datetime(2026, 8, 6, 12, tzinfo=UTC)
pool_by_id = {player.player_id: player for player in fixture_players}
start_event = DraftEvent(
    session_id="phase6-notebook",
    sequence=0,
    event_id="event-start",
    event_type="session_started",
    occurred_at=EVENT_TIME,
    command_id="command-start",
    payload={
        "rules": fixture_rules.model_dump(mode="json"),
        "ruleset_fingerprint": fixture_rules.fingerprint(),
        "user_draft_slot": 3,
        "projection_run_id": "synthetic-phase4",
        "adp_build_fingerprint": "synthetic-phase5",
        "player_pool_fingerprint": player_pool_fingerprint(fixture_players),
        "engine_config_fingerprint": engine_config.fingerprint(),
        "random_seed": 42,
        "simulation_count": 16,
    },
)
state = apply_event(None, start_event)
start_event = replace(start_event, resulting_state_fingerprint=state.fingerprint())
events = [start_event]
for sequence, player_id in enumerate(("WR-14", "RB-10"), start=1):
    player = pool_by_id[player_id]
    overall_pick = state.current_overall_pick
    team_id = state.current_team_id
    assert overall_pick is not None and team_id is not None
    event = DraftEvent(
        session_id=state.session_id,
        sequence=sequence,
        event_id=f"event-pick-{sequence}",
        event_type="pick_made",
        occurred_at=EVENT_TIME,
        command_id=f"command-pick-{sequence}",
        prior_state_fingerprint=state.fingerprint(),
        payload={
            "overall_pick": overall_pick,
            "team_id": team_id,
            "player_id": player.player_id,
            "player_name": player.display_name,
            "position": player.position,
            "projected_points": player.p50,
        },
    )
    state = apply_event(state, event)
    event = replace(event, resulting_state_fingerprint=state.fingerprint())
    events.append(event)

replayed_state = replay_events("phase6-notebook", tuple(events))
assert replayed_state == state
assert replayed_state.current_overall_pick == 3 and replayed_state.is_user_turn
pd.DataFrame(
    [
        ("events replayed", len(events)),
        ("current overall pick", replayed_state.current_overall_pick),
        ("next user pick after current", replayed_state.next_user_pick(include_current=False)),
        ("state fingerprint", replayed_state.fingerprint()),
    ],
    columns=["state fact", "value"],
)

## Run low-path deterministic recommendations

The engine shortlists legal candidates, computes ruleset replacement value and conditional next-turn availability, simulates the rest of the draft, and returns three distinct roles. Every score exposes its raw, normalized, weighted components.

In [ ]:
from fantasy_draft_ai.recommendations.engine import generate_recommendations

recommendations = generate_recommendations(
    replayed_state, fixture_players, engine_config
)
assert recommendations.available, recommendations.message
assert [candidate.role for candidate in recommendations.candidates] == [
    "balanced",
    "safe_floor",
    "high_upside",
]
assert len({candidate.player_id for candidate in recommendations.candidates}) == 3
pd.DataFrame(
    [
        {
            "role": candidate.role,
            "player_id": candidate.player_id,
            "position": candidate.position,
            "p10": candidate.p10,
            "p50": candidate.p50,
            "p90": candidate.p90,
            "replacement": candidate.replacement_points,
            "p50_vorp": candidate.p50_vorp,
            "P(available next)": candidate.probability_available_next_pick,
            "simulation_mean": candidate.simulation["mean_final_roster_value"],
            "recommendation_score": candidate.draft_recommendation_score,
        }
        for candidate in recommendations.candidates
    ]
)

In [ ]:
score_audit = []
for candidate in recommendations.candidates:
    recomputed = max(
        0.0,
        min(
            100.0,
            100.0 * sum(component.weighted_contribution for component in candidate.components),
        ),
    )
    score_audit.append(
        {
            "role": candidate.role,
            "displayed_score": candidate.draft_recommendation_score,
            "recomputed_score": recomputed,
            "matches": abs(candidate.draft_recommendation_score - recomputed) < 1e-9,
        }
    )
repeat = generate_recommendations(
    replayed_state, tuple(reversed(fixture_players)), engine_config
)
assert repeat == recommendations
assert repeat.fingerprint() == recommendations.fingerprint()
assert all(row["matches"] for row in score_audit)
pd.DataFrame(score_audit)

## Point-only uncertainty is not safety

A validated interval can be sampled asymmetrically around P50. A point-only baseline or rookie has `P10 = P50 = P90` and stays deterministic in simulation. Its zero width means uncertainty is unavailable, so it cannot silently win the safer-floor role.

In [ ]:
interval_example = fixture_players[0]
point_only_example = replace(
    fixture_players[-1],
    p10=fixture_players[-1].p50,
    p90=fixture_players[-1].p50,
    prediction_status="synthetic_point_only_fixture",
    projection_source="baseline",
)
remaining_selections = replayed_state.total_picks - len(replayed_state.picks)
work_units = (
    engine_config.candidate_count
    * replayed_state.simulation_count
    * remaining_selections
)
assert interval_example.has_outcome_interval
assert not point_only_example.has_outcome_interval
assert work_units <= engine_config.work_budget
pd.DataFrame(
    [
        ("interval row sampled", interval_example.has_outcome_interval),
        ("point-only row sampled", point_only_example.has_outcome_interval),
        ("candidate count", engine_config.candidate_count),
        ("simulation paths", replayed_state.simulation_count),
        ("remaining selections", remaining_selections),
        ("deterministic work units", work_units),
        ("configured work budget", engine_config.work_budget),
    ],
    columns=["assumption", "value"],
)

## Interpretation boundary

The three roles are transparent optimization baselines: balanced, safer floor, and higher upside. Their configured weights are displayed assumptions, and the opponent-pick model is uncalibrated. The fixture proves reproducibility and contract behavior; it does not validate real draft accuracy. No field represents championship probability because this engine does not model waivers, weekly lineups, matchups, playoffs, or calibrated team outcomes.